# 05 · Offline RL weights for ACT (gated stretch goal)

Filtered BC (ACT on `ours_all`) keeps only successful rollouts and treats all their frames as equally good. Here an IQL critic, trained on **all** generated rollouts (successes and failures) with the simulator's low-dim state, scores every frame by its advantage; ACT is then retrained on `ours_mix` with loss weights `exp(beta * A)` (see `p2p/rl/iql.py`).

**Gate** (decided before looking at the policy result): continue to the ACT run only if the critic is informative, i.e. its value half-way through an episode separates successes from failures (`auroc_V_mid_episode >= 0.75`). Otherwise the weights would be noise and the run would only cost GPU time.

The critic is small (MLPs on 18-D states): a few minutes on any runtime.

In [ ]:
import os, sys, json, subprocess
os.environ["MUJOCO_GL"] = "egl"                      # GPU rendering, must be set before mujoco is imported
REPO_URL = "https://github.com/LukasWill/phone2panda.git"
BASE = "/content" if os.path.exists("/content") else "/kaggle/working"   # Colab or Kaggle
CODE_DIR = f"{BASE}/phone2panda"
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q"], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[libero]==0.6.1", "imageio-ffmpeg"],
                   capture_output=True, text=True)
if r.returncode != 0:                                  # show pip's own reason, not just "exit status 1"
    print((r.stdout + r.stderr)[-4000:]); raise RuntimeError("pip install failed: see the lines above")
# Colab preinstalls an older torchao; a newer diffusers (pulled in by lerobot) then fails on import
# (FqnToConfig). ACT does not use torchao, and diffusers skips it when it is absent.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
from p2p.sim.env import ensure_nvidia_egl
ensure_nvidia_egl()      # Colab: make EGL render on the GPU (otherwise it silently renders on the CPU, ~10x slower)
# Hugging Face token (Colab: key icon -> Secrets -> HF_TOKEN; Kaggle: Add-ons -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: uploads to the Hub will fail")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    print("Hugging Face account:", whoami(token=os.environ["HF_TOKEN"])["name"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
HUB_REPO = "phone2panda-data"
from p2p.hub import pull, push
if not os.path.exists(f"{BASE}/generated"):
    pull(HUB_REPO, f"{BASE}/hub", allow_patterns=["raw/generated.tar", "lerobot/ours_mix/meta/info.json"])
    !tar -xf {BASE}/hub/raw/generated.tar -C {BASE}

In [ ]:
!python -m p2p.rl.iql {BASE}/generated {BASE}/rl --steps 50000 --beta 1.0

In [ ]:
import numpy as np
diag = json.load(open(f"{BASE}/rl/iql_diagnostics.json"))
w = np.load(f"{BASE}/rl/weights.npz")["weights"]
n_mix = json.load(open(f"{BASE}/hub/lerobot/ours_mix/meta/info.json"))["total_frames"]
assert len(w) == n_mix, f"weights ({len(w)}) and ours_mix ({n_mix}) frames differ: rebuild ours_mix from the same generated/ folder"
GATE = diag["auroc_V_mid_episode"] >= 0.75
print("critic informative:", GATE, diag)

In [ ]:
if GATE:
    push(f"{BASE}/rl", HUB_REPO, path_in_repo="rl")
    print('now run notebook 03 with DATASET = "ours_mix", WEIGHTS = "rl/weights.npz"')
else:
    print("gate not passed: report the diagnostics, skip the weighted ACT run")